# Test based on https://github.com/NathanWycoff/prox_alasso/blob/main/python/ada_sim.py

In [2]:
from sklearn.datasets import fetch_openml,load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV,LogisticRegressionCV 
from adaptivelasso.skcompat import MAPAdaptiveLasso
from adaptivelasso.utils import generate_data, report_model_results
import numpy as np
import time

## Compare MAPAdaptiveLasso with LassoCV

In [2]:
# Example for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2  # 5 informative features
sigma = 1.0
lik = 'normal'

X, y = generate_data(lik, N, P, beta, sigma=sigma, random_state=42)

# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=5000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

# Use a pipeline for scaling and LassoCV
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=0, max_iter=5000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

ES stop!
ES stop!
Estimated coefficients: [ 1.34543251e+00  1.34544432e+00  1.22827788e+00  1.36146619e+00
  1.36443837e+00  3.38394358e-02  1.65747310e-01  1.51566283e-01
  6.55682827e-02 -2.99462744e-02 -9.66660774e-02  1.14003002e-01
  7.85721607e-02  1.30524943e-01 -8.36840997e-02 -5.43380968e-02
  0.00000000e+00 -0.00000000e+00  1.37756660e-01 -3.16348111e-04]
R²: 0.5737
RMSE: 0.3241
Estimated coefficients: [ 0.16488544  0.1550696   0.14862804  0.16592588  0.16575047 -0.
  0.01683911  0.01516307  0.0040984  -0.0040288  -0.          0.01580763
 -0.          0.00102194  0.         -0.00028995  0.00109467  0.00600582
  0.01435396  0.        ]
R²: 0.4902
RMSE: 0.3544


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from adaptivelasso.skcompat import MAPAdaptiveLasso

# Assume generate_data and report_model_results are already defined as above

likelihoods = ['normal', 'cauchy', 'bernoulli', 'poisson', 'zinb', 'nb']
results = []
max_iters = 3000  # Set a maximum number of iterations
tol = 1e-4  # Set a tolerance for convergence
cv = 5

for lik in likelihoods:
    print(f"Testing likelihood: {lik}")
    N, P = 1000, 20
    true_coefs = np.zeros(P)
    true_coefs[:3] = 1.0
    sigma = 1.0

    X, y = generate_data(lik, N, P, beta=true_coefs, sigma=sigma, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    # MAPAdaptiveLasso
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=max_iters, track=True, tol=tol, cv=cv))
    ])
    pipe.fit(X_train, y_train)
    y_pred_adapt = pipe.predict(X_test)
    if lik == 'cauchy':
        residuals_adapt = y_test - y_pred_adapt
        mad_adapt = np.median(np.abs(residuals_adapt - np.median(residuals_adapt)))
        r2_adapt = r2_score(y_test, y_pred_adapt)
        rmse_adapt = np.nan
    else:
        r2_adapt = r2_score(y_test, y_pred_adapt)
        rmse_adapt = np.sqrt(mean_squared_error(y_test, y_pred_adapt))
        mad_adapt = np.nan

    # LassoCV
    pipe_lasso = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(cv=5, random_state=0, max_iter=max_iters))
    ])
    pipe_lasso.fit(X_train, y_train)
    y_pred_lasso = pipe_lasso.predict(X_test)
    if lik == 'bernoulli':
        y_pred_lasso_bin = (y_pred_lasso > 0.5).astype(int)
        r2_lasso = r2_score(y_test, y_pred_lasso_bin)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso_bin))
        mad_lasso = np.nan
    elif lik == 'cauchy':
        residuals_lasso = y_test - y_pred_lasso
        mad_lasso = np.median(np.abs(residuals_lasso - np.median(residuals_lasso)))
        r2_lasso = r2_score(y_test, y_pred_lasso)
        rmse_lasso = np.nan
    else:
        r2_lasso = r2_score(y_test, y_pred_lasso)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
        mad_lasso = np.nan

    # Decide which is better for each metric
    if lik == 'cauchy':
        better_mad = (
            'AdaptiveLasso' if mad_adapt < mad_lasso else
            'LassoCV' if mad_lasso < mad_adapt else
            'Tie'
        )
        better_rmse = np.nan
    else:
        better_mad = np.nan
        better_rmse = (
            'AdaptiveLasso' if rmse_adapt < rmse_lasso else
            'LassoCV' if rmse_lasso < rmse_adapt else
            'Tie'
        )

    if np.isnan(r2_lasso):
        better_r2 = 'AdaptiveLasso only'
    else:
        better_r2 = (
            'AdaptiveLasso' if r2_adapt > r2_lasso else
            'LassoCV' if r2_lasso > r2_adapt else
            'Tie'
        )

    results.append({
        'likelihood': lik,
        'R2_AdaptiveLasso': r2_adapt,
        'RMSE_AdaptiveLasso': rmse_adapt,
        'MAD_AdaptiveLasso': mad_adapt,
        'R2_LassoCV': r2_lasso,
        'RMSE_LassoCV': rmse_lasso,
        'MAD_LassoCV': mad_lasso,
        'Better_R2': better_r2,
        'Better_RMSE': better_rmse,
        'Better_MAD': better_mad,
    })

df_results = pd.DataFrame(results)
print(df_results)

Testing likelihood: normal
ES stop!
ES stop!
ES stop!
ES stop!
ES stop!
Testing likelihood: cauchy
ES stop!
Testing likelihood: bernoulli
ES stop!
ES stop!
Testing likelihood: poisson
ES stop!
ES stop!
ES stop!
Testing likelihood: zinb
ES stop!
ES stop!
Testing likelihood: nb
ES stop!
  likelihood  R2_AdaptiveLasso  RMSE_AdaptiveLasso  MAD_AdaptiveLasso  \
0     normal          0.722259            0.961862                NaN   
1     cauchy          0.020460                 NaN           0.928932   
2  bernoulli          0.192079            0.448612                NaN   
3    poisson          0.973212            2.188954                NaN   
4       zinb          0.352073           11.999097                NaN   
5         nb          0.506795           13.704425                NaN   

   R2_LassoCV  RMSE_LassoCV  MAD_LassoCV      Better_R2    Better_RMSE  \
0    0.736476      0.936920          NaN        LassoCV        LassoCV   
1    0.000429           NaN     1.374673  AdaptiveLass

In [3]:
lik='bernoulli'
X, y = fetch_openml(name="spambase", version=1, as_frame=False, return_X_y=True)
y = y.astype(int)
# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=3000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik, )

# Use a pipeline for scaling and LogisticRegressionCV 
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LogisticRegressionCV(cv=5, random_state=0, max_iter=3000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

Estimated coefficients: [ 0.01371693 -0.10502636  0.10890062  0.15293201  0.31134353  0.11461434
  0.72798937  0.25337844  0.21538928  0.09576328  0.24603989 -0.10822488
  0.0234595   0.02853528  0.25374314  0.3783016   0.39546022  0.13669363
  0.16798051  0.3106886   0.38644088  0.28757871  0.58078085  0.28543498
 -0.55514014 -0.35537384 -0.47549181  0.03895077 -0.18321147 -0.20661275
 -0.11431154 -0.05288002 -0.30013174 -0.04707865 -0.19940585  0.07981953
 -0.16449048 -0.04488241 -0.19788419 -0.05594981 -0.18627736 -0.32741445
 -0.11634778 -0.23579368 -0.35649626 -0.35373609 -0.1420299  -0.21174787
 -0.22875533 -0.06405651 -0.04529077  0.40921787  0.73291523  0.17747353
  0.20026912  0.36593683  0.39751947]
Accuracy: 0.4452
AUC: 0.7474
LogLoss: 12.5802
F1 Score: 0.5998
Estimated coefficients: [[-8.19332373e-02 -1.60969459e-01  8.03757702e-02  2.69109169e+00
   4.11328349e-01  2.08832889e-01  9.30981947e-01  2.01652395e-01
   1.90179103e-01  4.88075511e-02  3.22550322e-02 -1.02124357e

In [ ]:
lik='bernoulli'
X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True)
y = (y == '0').astype(int)
# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=200, track=False, tol=1e-4, cv=5))
])
start = time.time()
pipe.fit(X_train, y_train)
end = time.time()
# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik, )
print(f"MAPAdaptiveLasso training time: {end - start:.2f} seconds")

# Use a pipeline for scaling and LogisticRegressionCV 
mask = np.isin(y.astype(str), ['0', '1'])
X, y = X[mask], y[mask].astype(int)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear',random_state=0, max_iter=200, tol=1e-4))
])
start = time.time()
pipe_lassocv.fit(X_train, y_train)
end = time.time()
# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)
print(f"LogisticRegressionCV training time: {end - start:.2f} seconds")

Estimated coefficients: [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  2.63839377e-03  2.63839377e-03  2.63839377e-03  2.63839377e-03
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  2.35272050e-03  2.13664252e-03  6.53275103e-04
  3.68002245e-04 -1.64948426e-03 -2.49383886e-03  1.07793596e-03
  9.95819796e-04  2.87058057e-04 -1.43424501e-03 -1.37127693e-03
  6.32434944e-06 -9.27506533e-04 -1.27837816e-03 -7.60560297e-04
 -9.32218025e-04  3.07090975e-03  2.85321460e-03 -1.15230283e-04
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00 -1.89333756e-03
 

/Users/linpei/miniforge3/envs/tf_m1/lib/python3.8/site-packages/sklearn/svm/_base.py:1225: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/linpei/miniforge3/envs/tf_m1/lib/python3.8/site-packages/sklearn/svm/_base.py:1225: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/linpei/miniforge3/envs/tf_m1/lib/python3.8/site-packages/sklearn/svm/_base.py:1225: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [ ]:
## 569, 30
lik='bernoulli'
X, y = load_breast_cancer(return_X_y=True)
# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=3000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik, )

# Use a pipeline for scaling and LogisticRegressionCV 
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear',random_state=0, max_iter=3000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

Estimated coefficients: [-0.43284658 -0.44823709 -0.41690991 -0.39504691 -0.17765853  0.
 -0.32903955 -0.4783077  -0.12293695  0.19836532 -0.55304806 -0.04681364
 -0.3886881  -0.38223496  0.06158617  0.16657629  0.         -0.05051721
  0.03095343  0.24203437 -0.5484851  -0.59570487 -0.49448513 -0.47580604
 -0.35039828 -0.16931136 -0.40847688 -0.59091943 -0.28646175 -0.03868958]
Accuracy: 0.4123
AUC: 0.9082
LogLoss: 20.2991
F1 Score: 0.0000
Estimated coefficients: [[ 0.          0.          0.          0.          0.          0.
   0.         -0.51714952  0.          0.         -1.87942781  0.
   0.          0.          0.          0.          0.          0.
   0.          0.39213346 -2.33794032 -1.04845145 -0.08368801 -0.64001962
  -0.33046702  0.         -0.38619299 -1.31205453 -0.40123991  0.        ]]
Accuracy: 0.4123
AUC: 0.9803
LogLoss: 20.2991
F1 Score: 0.0000
